# Analysis of Survey Responses using LLM for Labelling

This notebook contains code that can label survey responses.

It is designed to be ran in the NHS Federated Data Platform.

## Method

### 1 - Setup

First, we will import the required modules.

In [ ]:
from foundry.transforms import Dataset
import pandas as pd
import numpy as np
import json
from string import Template
import textwrap
import tqdm
import time

from language_model_service_api.languagemodelservice_api_embeddings_v3 import GenericEmbeddingsRequest
from language_model_service_api.languagemodelservice_api_completion_v3 import GptChatCompletionRequest
from language_model_service_api.languagemodelservice_api import ChatMessage, ChatMessageRole
from palantir_models.models import OpenAiGptChatLanguageModel

from config import config, datasets
from src.llm import llm

class PromptTemplate(Template):
    delimiter = ""
    

Then, load our datasets.

In [ ]:
pd.set_option('display.max_colwidth', None)

for dataset_name, dataset in datasets.items():
    print(f"loading {dataset_name} ...", end="")
    dataset["input_data"] = Dataset.get(dataset["input_dataset_name"]).read_table()
    
    if config["test_mode"]["is_active"]:
        row_limit = config["test_mode"]["row_limit"]
        dataset["input_data"] = dataset["input_data"][0:row_limit]
        print(" done!")

model = OpenAiGptChatLanguageModel.get(config["open_ai_model"])

### 2 - Generate Prompts

In [ ]:
prompt_cleaning = """Clean this json file: {json_file}. Only output a json file, include no other text. The json should be of the format: {"Reasoning":"xx","Ideas for Change": "xx","Senitiment": "xx"}"""

prompt_intro = """
You identify 'Sentiment' and 'Ideas for Change' from a survey response. 

You are to note on 'Ideas for Change' and 'Sentiment', alongside giving your 'Reasoning'.

Each of these fields is a free text field. Sentiment must be one of 'Positive', 'Neutral', 'Poor' or 'Very Poor'.

Your response must be valid json.

Here are two examples of key topics extracted from survey responses.
"""

prompt_ending = """
Here is the survey reponse you must label with topics:

Survey Response:
{survey_response}

Within the 'Reasoning' section of your json output, think step by step to reason each topic. This are is for you to outline your thoughts, consider if you have missed any key topics, and check your responses.
Then, score on 'Sentiment' and 'Ideas for Change'. Your response must be valid json.

Your response must be in the format:

{"Reasoning":"xx","Ideas for Change": "xx", "Sentiment": "xx"}

Do not put any personal identifiable information in your response. Include nothing else in your output.
"""

In [ ]:
for dataset in datasets.values():
    all_example_prompts_map = {}
    prompt_examples = ""
    for i, example_for_prompt in enumerate(dataset["examples_for_prompt"]):
        
        prompt_examples += textwrap.dedent(f"""
        Example {i + 1}:
        {{example_{i + 1}_prompt}}

        The output would be:
        {{example_{i + 1}_output}}        
        """)
        
        example_for_prompt["prompt"] = llm.row_to_prompt(dataset["input_data"].loc[i], dataset["columns_for_prompt"])
        this_example_prompt_map = { 
            f"example_{i + 1}_prompt": example_for_prompt["prompt"],
            f"example_{i + 1}_output": example_for_prompt["output"]
        }
        all_example_prompts_map = { **all_example_prompts_map, **this_example_prompt_map}
    
    full_prompt = PromptTemplate(prompt_intro + prompt_examples + prompt_ending)
    
    dataset["prompt"] = full_prompt.safe_substitute(
        all_example_prompts_map
    )

    dataset["prompt"] = PromptTemplate(dataset["prompt"])

### 3 - Generate Labels

In [ ]:
for dataset_name, dataset in datasets.items():
    
    responses = [example_for_prompt["output"] for example_for_prompt in dataset["examples_for_prompt"]]
    
    dataset_row_count = len(dataset["input_data"])
   
    print(f"Starting response labelling for {dataset_name} ...")
    
    for i in tqdm.tqdm(range(len(responses), dataset_row_count)):

        formatted_prompt = dataset["prompt"].safe_substitute(
            survey_response = llm.row_to_prompt(dataset["input_data"].iloc[i], dataset["columns_for_prompt"])
        )
   
        response = model.create_chat_completion(GptChatCompletionRequest([ChatMessage(ChatMessageRole.USER, formatted_prompt)], temperature = config["temperature"]))
        raw_content = response.choices[0].message.content  # Extract the raw content

        try: 
            json_content = json.loads(raw_content)
            responses.append(json_content)

        except:

            try:
                raw_content = raw_content.replace("```","")
                raw_content = raw_content.replace("json","")
                raw_content = raw_content.replace("\n","")
                json_content = json.loads(raw_content)
                responses.append(json_content)
            except:

                try:
                    cleaning_prompt = cleaning_prompt.safe_substitute(json_file = raw_content)
                    response = model.create_chat_completion(GptChatCompletionRequest([ChatMessage(ChatMessageRole.USER, cleaning_prompt)], temperature = config["temperature"]))
                    raw_content = response.choices[0].message.content  # Extract the raw content
                    json_content = json.loads(raw_content)
                    responses.append(json_content)
                except:
                    responses.append(f"Error: {raw_content}")
    
    print(f"   labels generated...")
    print(f"   Saving data...")
    df_output = dataset["input_data"].copy()
    df_output["labels"] = responses

    if config["test_mode"]["is_active"]:
        print(df_output["labels"])
    else:
        output_dataset = Dataset.get(dataset[f"output_dataset_name"])
        output_dataset.write_table(df_output)
    print(f"   Done!", end="\n\n\n")